In [ ]:
import cv2
import mediapipe as mp
import csv
import numpy as np
import os   

# ===============================
# CONFIGURACIÓN
# ===============================

LABEL = "A"
OUTPUT_FILE = "dataset_sequences.csv"
SEQUENCE_LENGTH = 30 # Número de frames por secuencia
NUM_SEQUENCES = 100 # Número total de secuencias a recolectar

# ===============================
# MEDIAPIPE
# ===============================

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# ===============================
# FUNCION LANDMARKS → VECTOR
# ===============================

def hand_to_feature_vector(hand_landmarks):

    coords = []

    for lm in hand_landmarks.landmark:
        coords.append(lm.x)
        coords.append(lm.y)
        coords.append(lm.z)

    return coords  # 63 valores

# ===============================
# CÁMARA
# ===============================

cap = cv2.VideoCapture(0)

dataset = []
sequence = []

print("Presiona Q para salir")

while cap.isOpened():

    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(rgb)

    if results.multi_hand_landmarks:

        hand_landmarks = results.multi_hand_landmarks[0]

        mp_drawing.draw_landmarks(
            frame,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS
        )

        features = hand_to_feature_vector(hand_landmarks)

        sequence.append(features)

        if len(sequence) == SEQUENCE_LENGTH:

            dataset.append(sequence)

            print(f"Secuencia guardada {len(dataset)}/{NUM_SEQUENCES}")

            sequence = []

    cv2.putText(
        frame,
        f"Label: {LABEL} | Samples: {len(dataset)}",
        (10,30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        2
    )

    cv2.imshow("Recolectando datos", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    if len(dataset) >= NUM_SEQUENCES:
        break

cap.release()
cv2.destroyAllWindows()

# ===============================
# GUARDAR DATASET
# ===============================

rows = []

for seq in dataset:

    flat = np.array(seq).flatten()

    row = list(flat)
    row.append(LABEL)

    rows.append(row)

with open(OUTPUT_FILE, "a", newline="") as f:

    writer = csv.writer(f)

    for r in rows:
        writer.writerow(r)

print("Dataset guardado")

Presiona Q para salir


AttributeError: module 'google.protobuf.message_factory' has no attribute 'GetMessageClass'

: 

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("dataset_sequences.csv", header=None)

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [ ]:
SEQUENCE_LENGTH = 30
FEATURES = 63

X = X.reshape(-1, SEQUENCE_LENGTH, FEATURES)

print(X.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

encoder = LabelEncoder()

y = encoder.fit_transform(y)

y = to_categorical(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()

model.add(LSTM(128, return_sequences=True, input_shape=(30,63)))
model.add(Dropout(0.3))

model.add(LSTM(64))
model.add(Dropout(0.3))

model.add(Dense(64, activation="relu"))

model.add(Dense(y.shape[1], activation="softmax"))

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=40,
    batch_size=32,
    validation_data=(X_test, y_test)
)

In [ ]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("sign_model_motion.tflite", "wb") as f:
    f.write(tflite_model)

In [ ]:
sequence = []

if results.multi_hand_landmarks:

    features = hand_to_feature_vector(hand_landmarks)

    sequence.append(features)

    if len(sequence) > 30:
        sequence.pop(0)

    if len(sequence) == 30:

        X = np.expand_dims(sequence, axis=0)

        prediction = model.predict(X)

        predicted_class = np.argmax(prediction)